In [1]:
from llama_index.llms.groq import Groq
from llama_index.core import Settings

Settings.llm = Groq(model="openai/gpt-oss-20b",
                    context_window=8000)
response = Settings.llm.complete("Explain the importance of low latency LLMs")
print(response)

/home/arthas/anaconda3/envs/tcc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Why Low‑Latency LLMs Matter

Large language models (LLMs) are powerful, but their usefulness is only as good as the speed with which they can respond. In many real‑world scenarios, a delay of even a few hundred milliseconds can turn a great idea into a frustrating user experience or, in safety‑critical systems, into a failure. Below is a concise guide to the key reasons low latency is essential, the contexts where it matters most, and the trade‑offs involved.

| Domain | Why latency is critical | Typical latency target | Consequence of high latency |
|--------|------------------------|------------------------|-----------------------------|
| **Conversational AI / Chatbots** | Users expect instant replies; a 1‑second lag feels “robotic.” | < 200 ms for single‑turn, < 500 ms for multi‑turn | Higher churn, lower engagement |
| **Voice assistants** | Voice‑to‑text + LLM + text‑to‑speech must feel natural. | < 300 ms end‑to‑end | “Thinking” pauses, user frustration |
| **Real‑time trans

In [2]:
from llama_index.core.agent.workflow import AgentWorkflow

def sum(a, b):
    """useful for sum numbers"""
    return a + b
    
def multiply(a, b):
    """useful for multiply numbers"""
    return a * b
    
agent = AgentWorkflow.from_tools_or_functions(
        [sum, multiply],
        llm=Settings.llm,
        system_prompt="You are a helpful assistant that can do calculations"
)

response = await agent.run("What is 7 + 5 * 3?")
print(response)

# But the model stills responds to other contexts
response = await agent.run("Which is the name of Germany capital?")
print(response)

22
The capital of Germany is **Berlin**.


In [7]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

path = "./models/nomic-ai-v1.5"
# downloading the model
#
# from sentence_transformers import SentenceTransformer
#
# model = SentenceTransformer(
#     "nomic-ai/nomic-embed-text-v1.5",
#     trust_remote_code=True
# )
# 
# model.save(path)

Settings.embed_model = HuggingFaceEmbedding(
    model_name=path,
    trust_remote_code=True
)

documents = SimpleDirectoryReader("data_test").load_data()

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(
    response_mode="refine"
)

async def search_document(query: str) -> str:
    """useful for search about country's presidents"""
    response = await query_engine.aquery(query)
    return str(response)

# It'll restrict the model on another level
system_prompt = """ 
    ever, EVER, use only the search_document to give responses.
    Responds naturally, without marks that you viewed a document.
"""

agent = AgentWorkflow.from_tools_or_functions(
    [search_document],
    llm=Settings.llm,
    system_prompt=system_prompt
)

async def main():
    # In the 'documents = SimpleDirectoryReader("data_test").load_data()',
    # the information inside is: The Mongolia president at 1988
    # was Arthas Brunnett Mangueira
    response = await agent.run("What was the name of the Mongolia president at 1988?",
                               max_iterations=10)
    print(response)

await main()

2026-05-08 16:23:49,808 - INFO - Loading SentenceTransformer model from ./models/nomic-ai-v1.5.
2026-05-08 16:23:50,651 - WARNING - <All keys matched successfully>
2026-05-08 16:23:51,175 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-08 16:23:51,569 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"
2026-05-08 16:23:51,759 - INFO - HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"


The president of Mongolia in 1988 was Arthas Brunnett Mangueira.
